# Apex Retail Intelligence — Bronze Layer Script

**Author:** Aashi Phulera
**Programme:** Celebal Technologies | CEI'26 Internship Programme — Major Project
**Layer:** Bronze (Delta Lake)
**Technology:** PySpark, Delta Lake, Unity Catalog

---

### Purpose
This notebook implements **Phase 3** of the Medallion pipeline:
- Reads Landing Parquet files (output of `01_Raw_Landing_Script`)
- Writes them into Bronze Delta tables using ACID-compliant Delta Lake format
- Injects an `ingested_at` timestamp column into every table for full audit trail
- Handles incremental loads as **separate append-only writes** — Bronze retains all raw history with no deduplication, per design

### Prerequisites
Requires `01_Raw_Landing_Script` to have run first, since this notebook reads from the Landing Parquet zone it produces.

### Independently Executable
This notebook reads directly from the Landing zone Parquet files and writes to named Delta tables — it does not depend on any in-memory state from other notebooks.

⚠️ **Note:** Since Bronze is intentionally append-only, re-running this notebook without first re-running Phase 1/2 will duplicate row counts. This is expected pipeline behavior, not a bug.

In [0]:
# ============================================
# PHASE 3: BRONZE LAYER
# Reads Landing Parquet files, writes to Bronze in Delta format,
# injects ingested_at timestamp, handles incremental as separate append.
# ============================================

from pyspark.sql.functions import current_timestamp

base_landing = "/Volumes/apex_retail/bronze/incoming_data/landing"

def load_to_bronze(dataset_name, load_type):
    """Reads landing Parquet, adds ingestion metadata, writes to Bronze Delta table."""
    df = spark.read.parquet(f"{base_landing}/{load_type}/{dataset_name}")
    
    # Metadata injection: full audit trail of ingestion time
    df = df.withColumn("ingested_at", current_timestamp())
    
    table_name = f"apex_retail.bronze.{dataset_name}_{load_type}"
    
    # Bronze is append-only, no deduplication needed at this stage
    df.write.format("delta").mode("append").saveAsTable(table_name)
    
    row_count = df.count()
    print(f"[BRONZE] {table_name}: {row_count} rows appended")

for name in ["customer", "product", "sales"]:
    for load_type in ["historical", "incremental"]:
        load_to_bronze(name, load_type)

print("\n✅ Phase 3 Bronze layer complete.")

# Verify all 6 Bronze tables exist
print("\n📋 Bronze tables created:")
display(spark.sql("SHOW TABLES IN apex_retail.bronze"))

[BRONZE] apex_retail.bronze.customer_historical: 1052 rows appended
[BRONZE] apex_retail.bronze.customer_incremental: 1053 rows appended
[BRONZE] apex_retail.bronze.product_historical: 1043 rows appended
[BRONZE] apex_retail.bronze.product_incremental: 1041 rows appended
[BRONZE] apex_retail.bronze.sales_historical: 1002 rows appended
[BRONZE] apex_retail.bronze.sales_incremental: 1000 rows appended

✅ Phase 3 Bronze layer complete.

📋 Bronze tables created:


database,tableName,isTemporary
bronze,customer_historical,false
bronze,customer_incremental,false
bronze,product_historical,false
bronze,product_incremental,false
bronze,sales_historical,false
bronze,sales_incremental,false
